In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Product catalog

A scraped e-commerce export landed in your inbox. The SKU column is supposed to be unique, but something is off.

Quick reminder — `df.duplicated(keep=False)` marks **every** row involved in a duplicate, not just the second one.

Answer these questions, then clean:

- How many exact duplicate rows are there? How many distinct SKUs appear more than once?
- Look closely at the rows where the same SKU appears twice. Are they truly identical, or is something different between them?
- Clean the data: drop the true duplicate; for the SKU with conflicting prices, keep the higher price. Standardize `category` to lowercase and `in_stock` to boolean.
- After cleaning, use `np.unique` to confirm every SKU appears exactly once.

In [51]:
products = pd.DataFrame({
    'sku':      ['A100','B200','C300','A100','D400','E500','B200','F600'],
    'name':     ['Wireless Mouse','USB Hub','Desk Lamp','Wireless Mouse',
                 'Keyboard','Monitor','USB Hub','Webcam'],
    'category': ['Electronics','Electronics','Home Office','Electronics',
                 'Electronics','Electronics','Electronics','Electronics'],
    'price':    [29.99, 45.00, 22.50, 29.99, 79.99, 299.00, 47.50, 89.99],
    'in_stock': ['True','False','True','True','False','True','False','True'],
})

# Your code here

print(sum(products.duplicated()),'exact duplicated row(s)')
print(len(products['sku'][products.duplicated(subset=['sku'])].unique()),'appear more than once')
ds = products[products.duplicated(subset=['sku'],keep=False)]
print('price are in different in the two B200')

d = products.drop_duplicates().copy()
p = d.groupby('sku')['price'].max()

d['price'] = d['sku'].map(p)
d = d.drop_duplicates()
d['category'] = d['category'].str.lower()
d['in_stock'] = d['in_stock'].replace({'True':True, 'False':False})

s = np.unique(d['sku'])
len(s) == len(d['sku'])

1 exact duplicated row(s)
2 appear more than once
price are in different in the two B200


/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_10302/3035575058.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d['in_stock'] = d['in_stock'].replace({'True':True, 'False':False})


True

---

## Level 2 — Clinic visit records

A medical clinic exported visit data for a quarterly review. It's messy in several ways — figure out the issues and clean before answering.

Known issues:
- One exact duplicate visit
- `dept`: inconsistent casing
- `wait_min`: stored as strings; some are `'n/a'`
- `cost`: stored as `'$120.00'` strings
- `followup`: `'Yes'`/`'YES'`/`'yes'`/`'No'`/`'NO'`/`'no'` — standardize to lowercase

After cleaning:

1. Which department has the highest average visit cost?
2. What fraction of visits have a follow-up scheduled?
3. Is there a relationship between patient age and visit cost? Compute the correlation with `np.corrcoef`.
4. What is the 90th percentile wait time (in minutes)? Use `np.nanpercentile`.
5. Which patient has the most visits in this dataset?

In [ ]:
visits = pd.DataFrame({
    'visit_id': ['V01','V02','V03','V04','V05','V06','V07','V08',
                 'V09','V10','V11','V12','V01'],
    'patient':  ['P001','P002','P003','P004','P005','P006','P007','P008',
                 'P003','P001','P009','P010','P001'],
    'dept':     ['CARDIOLOGY','oncology','Cardiology','Neurology','ONCOLOGY',
                 'cardiology','Neurology','CARDIOLOGY','Oncology','neurology',
                 'Cardiology','NEUROLOGY','CARDIOLOGY'],
    'wait_min': ['12','35','n/a','18','42','8','n/a','25',
                 '15','40','22','30','12'],
    'cost':     ['$120.00','$340.00','$95.00','$210.00','$480.00','$75.00',
                 '$180.00','$310.00','$95.00','$220.00','$150.00','$260.00','$120.00'],
    'age':      [54, 38, 62, 45, 71, 29, 58, 43, 62, 54, 67, 33, 54],
    'followup': ['Yes','NO','yes','No','YES','no','Yes','NO',
                 'yes','No','YES','no','Yes'],
})

# Your code here
v = visits.drop_duplicates().copy()
v['dept'] = v['dept'].str.lower()
v['wait_min'] = pd.to_numeric(v['wait_min'], errors='coerce')
v['cost'] = v['cost'].str.replace('$','',regex = False)
v['cost'] = pd.to_numeric(v['cost'], errors='coerce')
v['followup'] = v['followup'].str.lower()

print(v.groupby('dept')['cost'].mean().idxmax(),'has the highest average cost')
print((v['followup']=='yes').mean(),'has follow up scheduled')
cors = np.corrcoef(v['age'],v['cost'])[0,1]
print('correlation is: ', cors)
print('not correlated')

print(np.nanpercentile(v['wait_min'],90))

print(v.groupby('patient')['age'].count().idxmax(),'has the most visits')

oncology has the highest average cost
0.5 has follow up scheduled
correlation is:  0.057340483725213574
not correlated
40.2
P001 has the most visits


---

## Level 3 — Quarterly sales

Thirteen order records from four sales reps across four regions. One order is a duplicate. Everything else needs cleaning — no steps provided.

Answer these five questions:

1. Which sales rep generated the most total revenue?
2. What is the revenue breakdown by region? Which region leads?
3. What are the 25th and 75th percentile revenue values per order, across all orders? Use `np.percentile`.
4. Rank the reps from lowest to highest total revenue using `np.argsort`. Who ranks last?
5. How many reps appear in more than one region?

In [84]:
sales = pd.DataFrame({
    'order_id': ['O001','O002','O003','O004','O005','O006','O007',
                 'O008','O009','O010','O011','O012','O003'],
    'rep':      ['Sarah Kim','Tom Blake','sarah kim','Mike Ross','Emma Ford',
                 'Tom Blake','MIKE ROSS','Sarah Kim','Emma Ford','SARAH KIM',
                 'Tom Blake','Mike Ross','sarah kim'],
    'region':   ['North','South','North','East','West','South','East',
                 'NORTH','West','North','SOUTH','east','North'],
    'product':  ['Widget A','Widget B','Widget A','Widget C','Widget B',
                 'Widget C','Widget A','Widget B','Widget C','Widget A',
                 'Widget B','Widget A','Widget A'],
    'units':    [12, 8, 12, 15, 6, 10, 18, 9, 14, 11, 7, 20, 12],
    'revenue':  ['$1,200','$960','$1,200','$2,250','$720','$1,500','$2,700',
                 '$1,080','$2,100','$1,100','$840','$3,000','$1,200'],
    'quarter':  ['Q1','Q1','Q1','Q1','Q2','Q2','Q2','Q3','Q3','Q3','Q4','Q4','Q1'],
})

# Your code here

c = sales.drop_duplicates().copy()
c['rep'] = c['rep'].str.lower()
c['region'] = c['region'].str.lower()
c['product'] = c['product'].str.lower()
c['revenue'] = c['revenue'].str.replace('$','',regex = False)
c['revenue'] = c['revenue'].str.replace(',','',regex = False)
c['revenue'] = pd.to_numeric(c['revenue'])
c['quarter'] = c['quarter'].str.lower()
c['order_id'] = c['order_id'].str.upper()

print(c.groupby('rep')['revenue'].sum().idxmax(),'generated the most revenue')
print(c.groupby('region')['revenue'].sum())
print(c.groupby('region')['revenue'].sum().idxmax(), 'leads')
qs = np.percentile(c['revenue'], [25, 50, 75])
print(qs[0],qs[1])
gs = c.groupby('rep')['revenue'].sum()
print(gs.index[np.argsort(gs)])
print(gs.index[np.argsort(gs)][-1],'rank last')

print(c.groupby('rep')['region'].unique())
print('no rep appear in more than one region')

mike ross generated the most revenue
region
east     7950
north    4580
south    3300
west     2820
Name: revenue, dtype: int64
east leads
1050.0 1200.0
Index(['emma ford', 'tom blake', 'sarah kim', 'mike ross'], dtype='object', name='rep')
mike ross rank last
rep
emma ford     [west]
mike ross     [east]
sarah kim    [north]
tom blake    [south]
Name: region, dtype: object
no rep appear in more than one region
